# Pretraining

**Baseline:** learn from raw text; predict the next token; evaluate validation loss/perplexity.

**In progress:** response export and final evaluation are still TODO.

| Goal | Run |
|---|---|
| Train from scratch | Under **Run folder**, set `checkpoint_path = None` and comment out the latest-checkpoint line. Then **Run All**. |
| Resume training | Under **Run folder**, uncomment the latest-checkpoint line, then **Run All**. `num_epochs` adds more epochs. |
| Generate from a checkpoint | Under **Run folder**, uncomment the latest-checkpoint line. Run **Goal & config → Load model → Generate / predict**. |

Restart the kernel when starting a different experiment.

## 0 · Goal & config

- **Mac:** local `.venv` kernel; run from `Training/`.
- **Runpod:** keep [setup.ipynb](../Runpod/setup.ipynb) on the local kernel and run **Start**. Here, choose **Select Kernel → Select Another Kernel → Existing Jupyter Server**, paste the URL, then choose **Python 3**. The first code cell sets the folder; device should print `cuda`.
- **Resume on Runpod:** upload the checkpoint into this notebook's run folder using Jupyter's file browser.

In [ ]:
from pathlib import Path

# Use the copied project folder on Runpod.
pod_folder = Path("/workspace/LLM-fine-tuning/Training")
if pod_folder.is_dir():
    %cd {pod_folder}

#### Imports

In [ ]:
from datetime import datetime, timezone

import tiktoken
import torch
from torch.utils.data import DataLoader, Dataset

from reference.configs import gpt2_BASE_CONFIG, gpt2_model_configs
from reference.data import load_pretraining_data
from reference.generate import generate_and_print_sample
from reference.losses import calc_loss_loader
from reference.model_io import text_to_token_ids, token_ids_to_text
from reference.models.gpt2 import GPT2Model
from reference.plot import plot_losses
from reference.train import train_model_simple

#### Model config

In [ ]:
# Options: gpt2-small (124M), gpt2-medium (355M), gpt2-large (774M), gpt2-xl (1558M).
CHOOSE_MODEL = "gpt2-small (124M)"

# ** unpacks dictionary entries; later values win on duplicate keys.
model_config = {**gpt2_BASE_CONFIG, **gpt2_model_configs[CHOOSE_MODEL]}

# Short context leaves enough validation examples in this small text.
model_config["context_length"] = 256
start_context = "Hello, how are"

#### Run folder

One folder per experiment; checkpoint and response filenames share a UTC timestamp (same clock on Mac and Runpod).

In [ ]:
folder = Path("runs/pretraining/001_baseline")
checkpoint_path = None  # Start with random weights.
# checkpoint_path = max(folder.glob("checkpoint_*.pth"))  # Uncomment to load the latest checkpoint.

#### Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print("device:", device)

#### Tokenizer

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text_to_token_ids(start_context, tokenizer)

In [ ]:
token_ids_to_text(text_to_token_ids(start_context, tokenizer), tokenizer)

## 1 · Load model

Create a model with random weights, or restore a checkpoint's model and optimizer.

In [ ]:
%%capture

if checkpoint_path is not None:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    # Older checkpoints use the model config set above.
    model_config = checkpoint.get("model_config", model_config)

torch.manual_seed(123)
model = GPT2Model(model_config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
if checkpoint_path is not None:
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    responses_file = checkpoint_path.with_name(
        checkpoint_path.stem.replace("checkpoint_", "responses_") + ".json"
    )
model.eval()

## 2 · Data + DataLoader

Load data, prepare token IDs, and build batches.

In [ ]:
text = load_pretraining_data()
print(f"Length of text: {len(text)}")

### Split data: 90% train, 10% validation

In [ ]:
split_idx = int(0.9 * len(text))
train_data = text[:split_idx]
val_data = text[split_idx:]

### Dataset

Split text into fixed-length chunks. Each example returns inputs and next-token targets.

In [ ]:
class PreTrainingDataset(Dataset):

    def __init__(self, txt, tokenizer, max_length, stride):

        self.input_ids  = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk  = token_ids[i: i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

### Custom collate function

No custom function is needed: examples already have equal lengths, so DataLoader stacks them. `calc_loss_batch` moves batches to the device during training.

### DataLoader

Build the loaders and inspect one batch.

In [ ]:
def create_dataloader_v1(txt, max_length=256, stride=128, 
                        batch_size=4, shuffle=True, 
                        drop_last=True, num_workers=0):

    tokenizer  = tiktoken.get_encoding("gpt2")
    dataset    = PreTrainingDataset(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
        )
    
    return dataloader

In [ ]:
num_workers = 0
batch_size = 2
ctx = model_config["context_length"]

# Stride equals context length, so input chunks do not overlap.
train_loader = create_dataloader_v1(
    train_data, max_length=ctx, stride=ctx, batch_size=batch_size,
    num_workers=num_workers
)
val_loader = create_dataloader_v1(
    val_data, max_length=ctx, stride=ctx, batch_size=batch_size,
    shuffle=False, drop_last=False, num_workers=num_workers
)

# Inspect one batch.
inputs, targets = next(iter(train_loader))
print(f"inputs.shape:  {inputs.shape}\ntargets.shape: {targets.shape}")

## 3 · Loss / objective

Next-token cross-entropy over every target. Measure the starting loss before training.

In [ ]:
torch.manual_seed(123)
model.eval()
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("Training loss before training:", train_loss)
print("Validation loss before training:", val_loss)

## 4 · Generate / predict

Generate a starting response before this training run.

In [ ]:
generate_and_print_sample(model, tokenizer, device, start_context)

## 5 · Train

Run this cell again to continue with the current model and optimizer. Epoch/step counters restart each call.

In [ ]:
%%time

torch.manual_seed(123)
num_epochs = 10

train_losses, val_losses, tokens_seen = train_model_simple(
    model, optimizer, device, tokenizer,
    train_loader, val_loader,
    eval_freq=5, eval_iter=5,
    start_context=start_context, num_epoch=num_epochs
)

### Plot Losses

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

## 6 · Save checkpoint & responses

**Runpod → Mac:** download this run's files into the same run folder on your Mac before **Delete → Confirm** in setup (local kernel). Deleting the pod also deletes its files.

#### Save model & optimizer

In [ ]:
folder.mkdir(parents=True, exist_ok=True)

latest = folder / f"checkpoint_{datetime.now(timezone.utc):%Y%m%d_%H%M%S_%f}.pth"
checkpoint = {
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "model_config": model_config,
}
torch.save(checkpoint, latest)
responses_file = latest.with_name(latest.stem.replace("checkpoint_", "responses_") + ".json")

#### Extract model responses & save

In [ ]:
# TODO: generate fixed prompts and save the responses to responses_file.

## 7 · Evaluate & reflect

Compare validation loss/perplexity with the starting model. Note what changed.

In [ ]:
# TODO: measure final validation loss and perplexity.